In [ ]:
import json
import random
import subprocess
from pathlib import Path
from tqdm import tqdm

import cv2

root = Path("..").resolve()
pictures = Path("pictures")
pictures.mkdir(exist_ok=True)
rng = random.Random(42)
query = 'SELECT DISTINCT f.file_name FROM dbo.record r JOIN cc_system.files f ON f.id = r.filename ORDER BY f.file_name'
videos = subprocess.check_output([str(root / "db.sh"), "psql", "-At", "-c", query], text=True).splitlines()
rng.shuffle(videos)

In [ ]:
frames = []
for video in tqdm(videos):
    cap = cv2.VideoCapture(str(root / "sample-500/files" / video))
    index = rng.randrange(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
    cap.set(cv2.CAP_PROP_POS_FRAMES, index)
    _, frame = cap.read()
    cap.release()
    name = f"{Path(video).stem}_{index}.png"
    cv2.imwrite(str(pictures / name), frame)
    frames.append({"file": name, "quality_ok": None})

Path("frames.json").write_text(json.dumps(frames, indent=2), encoding="utf-8")
len(frames)